In [ ]:
"""
Convert a CSV feature file to Weka's ARFF format.

Usage:
    python csv_to_arff.py features.csv
    python csv_to_arff.py features.csv output.arff

The first line of the CSV must be the attribute (feature) names.
Columns where every value is a number become NUMERIC attributes.
Any other column (e.g. the class label) becomes a NOMINAL attribute
listing all of its values. Empty cells are written as Weka's missing value "?".
"""
import csv
import os
import sys


def is_number(value):
    try:
        float(value)
        return True
    except ValueError:
        return False


def quote(text):
    """Quote names/values that contain spaces or characters ARFF treats specially."""
    if text == "" or any(c in text for c in " ,{}%'\"\t"):
        return "'" + text.replace("\\", "\\\\").replace("'", "\\'") + "'"
    return text


def main():
    if len(sys.argv) < 2:
        print("Usage: python csv_to_arff.py input.csv [output.arff]")
        sys.exit(1)

    csv_path = sys.argv[1]
    arff_path = sys.argv[2] if len(sys.argv) > 2 else os.path.splitext(csv_path)[0] + ".arff"

    with open(csv_path, newline="") as f:
        rows = [row for row in csv.reader(f) if row]

    header = [name.strip() for name in rows[0]]
    data = [[value.strip() for value in row] for row in rows[1:]]

    for i, row in enumerate(data, start=2):
        if len(row) != len(header):
            sys.exit(f"Line {i} has {len(row)} values but the header has {len(header)} names.")

    attributes = []
    for col, name in enumerate(header):
        values = [row[col] for row in data if row[col] not in ("", "?")]
        if values and all(is_number(v) for v in values):
            attributes.append(f"@attribute {quote(name)} numeric")
        else:
            unique = sorted(set(values))
            attributes.append(f"@attribute {quote(name)} {{{','.join(quote(v) for v in unique)}}}")

    relation = os.path.splitext(os.path.basename(csv_path))[0]
    with open(arff_path, "w", newline="\n") as out:
        out.write(f"@relation {quote(relation)}\n\n")
        out.write("\n".join(attributes) + "\n\n@data\n")
        for row in data:
            out.write(",".join("?" if v in ("", "?") else quote(v) for v in row) + "\n")

    print(f"Wrote {arff_path} ({len(data)} rows, {len(header)} attributes)")


if __name__ == "__main__":
    main()